# DarkTech Remote Worker (Colab GPU)

Roda o worker de geração de áudio na GPU do Colab e expõe via Cloudflare Tunnel.

Sua UI Gradio local (no VSCode) chama essa URL quando precisa renderizar um stem.
Mastering, orquestração com DeepSeek e a UI ficam locais.

**Antes de rodar**: Runtime > Change runtime type > GPU (T4 grátis basta; A100 com Pro).

In [ ]:
# 1/4: API key para esta sessão
import os, secrets

os.environ['DARKTECH_WORKER_API_KEY'] = secrets.token_urlsafe(24)
print('API key desta sessão:')
print('   ', os.environ['DARKTECH_WORKER_API_KEY'])
print()
print('Copie para o seu .env local como DARKTECH_REMOTE_API_KEY')

os.environ.setdefault('HF_TOKEN', '')  # opcional, sobe limites de download HF

In [ ]:
# 2/4: clonar o repo, instalar deps (incluindo ACE-Step), montar Drive, instalar cloudflared
import os, subprocess, shutil, sys, importlib
from pathlib import Path

REPO_URL = 'https://github.com/horningwalter/gorit-lab-darktech-generator.git'
REPO_DIR = Path('/content/gorit_lab_darktech_generator')

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '-q', 'origin'])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'reset', '--hard', '-q', 'origin/main'])
%cd /content/gorit_lab_darktech_generator
subprocess.check_call(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'])

print('\nInstalando darktech-generator + extras ML (~1-2 min)...')
res = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '.[ml]',
     'fastapi', 'uvicorn[standard]'],
    capture_output=True, text=True,
)
if res.returncode != 0:
    print('PIP FALHOU. stdout:')
    print(res.stdout[-3000:])
    print('stderr:')
    print(res.stderr[-3000:])
    raise SystemExit(1)
print('darktech-generator OK.')

# ACE-Step é instalado SEPARADAMENTE (depois) porque tem dependências com pinos
# rígidos (gradio==5.23, librosa==0.11, transformers==4.50) que conflitariam com
# o resolver se estivessem no nosso pyproject. Aqui deixamos o pip resolver no
# fim, sobrescrevendo o que precisar.
print('\nInstalando ACE-Step do GitHub (~2-4 min)...')
res = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'git+https://github.com/ace-step/ACE-Step.git'],
    capture_output=True, text=True,
)
if res.returncode != 0:
    print('PIP (ACE-Step) FALHOU. stdout:')
    print(res.stdout[-3000:])
    print('stderr:')
    print(res.stderr[-3000:])
    raise SystemExit(1)
print('ACE-Step OK.')

# Fallback à prova de cache do kernel
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

try:
    import darktech_generator
    print('\ndarktech_generator importado de', darktech_generator.__file__)
    print('versão', darktech_generator.__version__)
except Exception as e:
    print('IMPORT darktech FALHOU:', e)
    raise

# Sanity check: o módulo do ACE-Step é importável?
for candidate in ('acestep.pipeline_ace_step', 'acestep.pipeline',
                  'ace_step.pipeline_ace_step', 'ace_step.pipeline'):
    try:
        __import__(candidate)
        print(f'ACE-Step pipeline disponível em: {candidate}')
        break
    except ImportError:
        continue
else:
    print('AVISO: nenhum candidato de import do ACE-Step funcionou.')
    print('Conteúdo de site-packages (procurando ace):')
    sp_dirs = [p for p in sys.path if p.endswith('site-packages')]
    for d in sp_dirs:
        for entry in Path(d).glob('ace*'):
            print('  ', entry)

from darktech_generator.colab.bootstrap import bootstrap
import json
print('\nBootstrap:')
print(json.dumps(bootstrap(), indent=2))

if not shutil.which('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cf.deb
    !sudo dpkg -i /tmp/cf.deb 2>&1 | tail -2
print('cloudflared:', shutil.which('cloudflared'))

In [ ]:
# 3/4: subir o worker FastAPI em background e checar /health
# Se o worker já estava rodando de uma execução anterior, matá-lo primeiro.
import subprocess, sys, time, urllib.request, json, os, signal
from pathlib import Path

REPO_DIR = Path('/content/gorit_lab_darktech_generator')
worker_log = Path('/tmp/worker.log')

# mata worker anterior se existir
try:
    subprocess.run(['pkill', '-f', 'uvicorn.*remote_worker'], check=False)
    time.sleep(2)
except Exception:
    pass

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src') + os.pathsep + env.get('PYTHONPATH', '')

worker = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn',
     'darktech_generator.generation.remote_worker:build_app',
     '--factory', '--host', '127.0.0.1', '--port', '8000'],
    stdout=worker_log.open('w'), stderr=subprocess.STDOUT, env=env,
)
print('Worker PID', worker.pid, '(logs em /tmp/worker.log)')

ok = False
for i in range(40):
    if worker.poll() is not None:
        print('WORKER MORREU. Últimas linhas do log:')
        print(worker_log.read_text()[-2000:])
        raise SystemExit(1)
    try:
        req = urllib.request.Request(
            'http://127.0.0.1:8000/health',
            headers={'X-API-Key': os.environ['DARKTECH_WORKER_API_KEY']},
        )
        with urllib.request.urlopen(req, timeout=2) as r:
            print('/health:', json.dumps(json.loads(r.read()), indent=2))
            ok = True
            break
    except Exception:
        time.sleep(1)
if not ok:
    print('Worker não respondeu em 40s. Últimas linhas do log:')
    print(worker_log.read_text()[-2000:])
    raise SystemExit(1)
print('Worker pronto em http://127.0.0.1:8000')

In [ ]:
# 4/4: subir Cloudflare Tunnel e capturar URL pública
# Deixe esta célula rodando enquanto usa a UI local. Para encerrar, pare a célula.
import subprocess, re, time
from pathlib import Path

tunnel_log = Path('/tmp/cloudflared.log')
if tunnel_log.exists():
    tunnel_log.unlink()

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate',
     '--url', 'http://127.0.0.1:8000',
     '--logfile', str(tunnel_log),
     '--loglevel', 'info'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url_re = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
public_url = None

print('Aguardando URL do Cloudflare (até 60s)...', flush=True)
deadline = time.time() + 60
while public_url is None and time.time() < deadline:
    if tunnel.poll() is not None:
        print('cloudflared morreu. Log:')
        print(tunnel_log.read_text()[-2000:] if tunnel_log.exists() else '(sem log)')
        raise SystemExit(1)
    line = tunnel.stdout.readline()
    if line:
        print(line.rstrip(), flush=True)
        m = url_re.search(line)
        if m:
            public_url = m.group(0)
            break
    else:
        if tunnel_log.exists():
            m = url_re.search(tunnel_log.read_text())
            if m:
                public_url = m.group(0)
                break
        time.sleep(0.3)

if public_url is None:
    print('Não consegui extrair a URL em 60s. Conteúdo do log:')
    print(tunnel_log.read_text() if tunnel_log.exists() else '(sem log)')
    raise SystemExit(1)

print('\n=====================================================')
print(' Cloudflare Tunnel pronto')
print(' Cole no .env local da sua máquina:')
print(f'   DARKTECH_REMOTE_URL={public_url}')
print('=====================================================\n')
print('Mantendo célula viva. Pare a execução para encerrar o tunnel.', flush=True)

try:
    for line in iter(tunnel.stdout.readline, ''):
        print(line.rstrip(), flush=True)
except KeyboardInterrupt:
    tunnel.terminate()
    print('Tunnel encerrado.')